# So sánh Pretrained vs Fine-tuned
So sánh model gốc (COCO) với model đã fine-tune trên dataset bóng đá.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
if not os.path.exists('/content/football_tracking'):
    !git clone https://github.com/henruysun2511/football_tracking.git /content/football_tracking
%cd /content/football_tracking
!pip install -q ultralytics pyyaml matplotlib pandas

In [ ]:
# Download dataset + model
from roboflow import Roboflow
from pathlib import Path

api_key = input("ROBOFLOW_API_KEY: ")
rf = Roboflow(api_key=api_key)

if not os.path.exists('datasets/football-players-detection-2'):
    project = rf.workspace("roboflow-jvuqo").project("football-players-detection-3zvbc")
    dataset = project.version(2).download("yolov8")
    !mv {dataset.location} datasets/football-players-detection-2

# Copy model từ Drive nếu có
if os.path.exists('/content/drive/MyDrive/football_models/player_detector.pt'):
    !cp /content/drive/MyDrive/football_models/player_detector.pt models/
    print("Copied from Drive")
elif not os.path.exists('models/player_detector.pt'):
    print("WARNING: No fine-tuned model found!")

In [ ]:
import os, yaml, warnings
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

warnings.filterwarnings('ignore')
ROOT = Path(os.getcwd())
OUTPUT_PNG = "/content/drive/MyDrive/football_models/comparison.png"
os.makedirs("/content/drive/MyDrive/football_models", exist_ok=True)

# ─── 1. Load 2 models ───
print("="*60)
print("  SO SÁNH MODEL: PRETRAINED (COCO) vs FINE-TUNED (Football)")
print("="*60)

# Pretrained COCO
print("\n[1] Loading pretrained YOLOv8x (COCO)...")
model_coco = YOLO("yolov8x.pt")  # tự download từ Ultralytics

# Fine-tuned
print("[2] Loading fine-tuned model...")
model_ft = YOLO("models/player_detector.pt")

In [ ]:
# ─── 2. So sánh kiến trúc ───
print("\n" + "-"*60)
print("  BẢNG 1 — SO SÁNH KIẾN TRÚC")
print("-"*60)

def model_info(m):
    s = str(m.model)
    layers = s.count('(')  # đếm gần đúng số layers
    params = sum(p.numel() for p in m.model.parameters())
    return layers, params

l1, p1 = model_info(model_coco)
l2, p2 = model_info(model_ft)

# GFLOPs từ summary
s1 = str(model_coco.model.summary())
s2 = str(model_ft.model.summary())
import re
g1 = re.search(r'([\d.]+) GFLOPs', s1)
g2 = re.search(r'([\d.]+) GFLOPs', s2)
gflops_coco = float(g1.group(1)) if g1 else 0
gflops_ft   = float(g2.group(1)) if g2 else 0

print(f"  {'Metric':<25} {'COCO Pretrained':<20} {'Fine-tuned':<20}")
print(f"  {'-'*25} {'-'*20} {'-'*20}")
print(f"  {'Number of classes':<25} {80:<20} {4:<20}")
print(f"  {'Parameters':<25} {p1/1e6:<20.2f}M {p2/1e6:<20.2f}M")
print(f"  {'GFLOPs':<25} {gflops_coco:<20.1f} {gflops_ft:<20.1f}")

In [ ]:
# ─── 3. Fix data.yaml ───
base = ROOT / "datasets" / "football-players-detection-2"
with open(base / "data.yaml") as f:
    data_cfg = yaml.safe_load(f)
data_cfg["train"] = str(base / "train" / "images")
data_cfg["val"]   = str(base / "valid" / "images")
fixed_yaml = str(base / "_data_fixed.yaml")
with open(fixed_yaml, "w") as f:
    yaml.dump(data_cfg, f)

In [ ]:
# ─── 4. Val pretrained COCO trên football dataset ───
# YOLO tự động thay đổi head để khớp 4 classes
# Sau val, model sẽ có head 4 classes
print("\n" + "-"*60)
print("  EVAL PRETRAINED COCO → Football Dataset")
print("-"*60)
results_coco = model_coco.val(data=fixed_yaml, imgsz=1280, batch=8, device=0, plots=False, verbose=False)

coco_map50    = float(results_coco.box.map50)
coco_map50_95 = float(results_coco.box.map)
coco_prec     = float(results_coco.box.p[0]) if hasattr(results_coco.box, 'p') and len(results_coco.box.p) > 0 else 0
coco_rec      = float(results_coco.box.r[0]) if hasattr(results_coco.box, 'r') and len(results_coco.box.r) > 0 else 0
coco_ap50     = results_coco.box.ap50.tolist() if hasattr(results_coco.box, 'ap50') else [0]*4

print(f"  mAP@0.5    : {coco_map50:.4f}")
print(f"  mAP@0.5:0.95: {coco_map50_95:.4f}")
print(f"  Precision  : {coco_prec:.4f}")
print(f"  Recall     : {coco_rec:.4f}")

In [ ]:
# ─── 5. Val fine-tuned model ───
print("\n" + "-"*60)
print("  EVAL FINE-TUNED → Football Dataset")
print("-"*60)
results_ft = model_ft.val(data=fixed_yaml, imgsz=1280, batch=8, device=0, plots=False, verbose=False)

ft_map50    = float(results_ft.box.map50)
ft_map50_95 = float(results_ft.box.map)
ft_prec     = float(results_ft.box.p[0]) if hasattr(results_ft.box, 'p') and len(results_ft.box.p) > 0 else 0
ft_rec      = float(results_ft.box.r[0]) if hasattr(results_ft.box, 'r') and len(results_ft.box.r) > 0 else 0
ft_ap50     = results_ft.box.ap50.tolist() if hasattr(results_ft.box, 'ap50') else [0]*4

print(f"  mAP@0.5    : {ft_map50:.4f}")
print(f"  mAP@0.5:0.95: {ft_map50_95:.4f}")
print(f"  Precision  : {ft_prec:.4f}")
print(f"  Recall     : {ft_rec:.4f}")

In [ ]:
# ─── 6. Bảng so sánh tổng hợp ───
print("\n" + "="*70)
print("  BẢNG 2 — SO SÁNH HIỆU NĂNG TRÊN FOOTBALL DATASET")
print("="*70)

classes = ["ball", "goalkeeper", "player", "referee"]

header = f"{'Metric':<25} {'COCO Pretrained':<20} {'Fine-tuned':<20} {'Δ':<10}"
sep = "  " + "-"*25 + " " + "-"*20 + " " + "-"*20 + " " + "-"*10
print(header)
print(sep)

def row(name, a, b):
    delta = b - a
    sign = "+" if delta > 0 else ""
    print(f"  {name:<25} {a:<20.4f} {b:<20.4f} {sign}{delta:<8.4f}")

row("mAP@0.5", coco_map50, ft_map50)
row("mAP@0.5:0.95", coco_map50_95, ft_map50_95)
row("Precision", coco_prec, ft_prec)
row("Recall", coco_rec, ft_rec)
print()

print(f"  {'Per-class mAP@0.5':<25} {'COCO Pretrained':<20} {'Fine-tuned':<20} {'Δ':<10}")
print(sep)
for i, name in enumerate(classes):
    delta = ft_ap50[i] - coco_ap50[i]
    sign = "+" if delta > 0 else ""
    print(f"  {name:<25} {coco_ap50[i]:<20.4f} {ft_ap50[i]:<20.4f} {sign}{delta:<8.4f}")
print("="*70)

In [ ]:
# ─── 7. Vẽ biểu đồ so sánh ───
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("YOLOv8x — COCO Pretrained vs Fine-tuned on Football Dataset",
             fontsize=13, fontweight="bold")

# (a) Overall metrics
ax = axes[0]
metrics = ["mAP@0.5", "mAP@0.5:0.95", "Precision", "Recall"]
coco_vals = [coco_map50, coco_map50_95, coco_prec, coco_rec]
ft_vals   = [ft_map50,   ft_map50_95,   ft_prec,   ft_rec]

x = np.arange(len(metrics))
w = 0.35
bars1 = ax.bar(x - w/2, coco_vals, w, label="COCO Pretrained", color="#95a5a6", edgecolor="white")
bars2 = ax.bar(x + w/2, ft_vals,   w, label="Fine-tuned (Football)", color="#2ecc71", edgecolor="white")

for bar, val in zip(bars2, ft_vals):
    delta = val - coco_vals[list(ft_vals).index(val)]
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"+{delta:.2f}" if delta > 0 else f"{delta:.2f}",
            ha="center", va="bottom", fontsize=9, fontweight="bold", color="#27ae60")

ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.15)
ax.legend(loc="lower right")
ax.grid(alpha=0.3, axis="y")
ax.set_title("Overall Metrics Comparison")

# (b) Per-class mAP@0.5
ax = axes[1]
classes_display = ["ball", "gk", "player", "ref"]
x = np.arange(len(classes_display))
bars1 = ax.bar(x - w/2, coco_ap50, w, label="COCO Pretrained", color="#95a5a6", edgecolor="white")
bars2 = ax.bar(x + w/2, ft_ap50,   w, label="Fine-tuned (Football)", color="#2ecc71", edgecolor="white")

for bar, val, coco_val in zip(bars2, ft_ap50, coco_ap50):
    delta = val - coco_val
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"+{delta:.2f}" if delta > 0 else f"{delta:.2f}",
            ha="center", va="bottom", fontsize=9, fontweight="bold", color="#27ae60")

ax.set_xticks(x)
ax.set_xticklabels(classes_display)
ax.set_ylim(0, 1.15)
ax.legend(loc="lower right")
ax.grid(alpha=0.3, axis="y")
ax.set_title("Per-class mAP@0.5")

plt.tight_layout()
plt.savefig(OUTPUT_PNG, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {OUTPUT_PNG}")

# Dọn
os.remove(fixed_yaml)